# Application Development

With the machine learning model completed, the next phase of the project focuses on transforming it into a real-world application.

The application retrieves real-time and forecasted information from external APIs. Real-time space weather data are prepared as inputs for the trained Random Forest model, while forecast and environmental data are used separately to estimate observation conditions for a selected location and date.

This notebook develops and tests each component of that pipeline before integrating them into the final application.

## External APIs

The application relies on several external APIs to retrieve real-time space weather data, forecast geomagnetic activity and local environmental conditions.

The APIs selected for this project are:

- **NOAA Space Weather API** – Current solar wind, magnetic field measurements, and 45-day space weather forecast products.
- **NOAA Solar Cycle Forecast API** – Forecasted smoothed sunspot number and solar cycle activity.
- **Open-Meteo API** – Cloud cover and visibility.
- **Sunrise-Sunset API** – Daylight and darkness calculations.
- **Geolocation API** – User latitude and longitude.

## NOAA Space Weather API

The NOAA Space Weather Prediction Center (SWPC) provides free access to both real-time and forecasted space weather observations.

Since the machine learning model was trained using solar wind and magnetic field measurements, this API supplies the majority of the physical variables required by the prediction engine.

The following sections retrieve these data directly from NOAA and prepare them for later integration with the prediction model.

Base URL:
https://services.swpc.noaa.gov/json/

Purpose:
Provides real-time and forecasted space weather data.

Endpoints used:

- rtsw/rtsw_wind_1m.json
  - Solar wind speed
  - Proton density
  - Temperature

- rtsw/rtsw_mag_1m.json
  - Bx
  - By
  - Bz

- 45-day-forecast.json
  - Long-term space weather forecast
 
- ovation_aurora_latest.json
    - Near-real-time auroral activity
    - Aurora intensity by latitude and longitude
    - Used to create the interactive auroral activity map

## Real-time Solar Wind

The NOAA SWPC API provides real-time measurements of the solar wind, including proton speed, proton density, and proton temperature. These variables are required by the prediction engine because they are part of the features used by the trained Random Forest model.

The following steps retrieve the latest observations and extract only the variables required for prediction.

In [1]:
import requests
import pandas as pd

In [2]:
url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_wind_1m.json"

response = requests.get(url)

data = response.json()

The API returns the information in JSON format. Converting it into a pandas DataFrame makes it easier to manipulate and extract the required variables.

In [3]:
df_wind = pd.DataFrame(data)

df_wind.head()

,time_tag,active,source,proton_speed,proton_temperature,proton_density,proton_vx_gse,proton_vy_gse,proton_vz_gse,proton_vx_gsm,...,alpha_vz_gsm,alpha_sample_size,max_convergence_flag,max_data_flag,max_error_count_flag,max_processing_flag,max_range_flag,max_sample_count_flag,max_telemetry_flag,overall_quality
0,2026-08-10T11:01:00,True,SOLAR1,414.60,138877,3.65,-412.1,13.1,-43.4,-412.1,...,None,None,0,0,0,0,0,0,0,0
1,2026-08-10T11:00:08,False,IMAP,406.78,107995,2.63,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
2,2026-08-10T11:00:00,True,SOLAR1,407.60,138389,3.18,-404.5,17.5,-46.5,-404.5,...,None,None,0,0,0,0,0,0,0,0
3,2026-08-10T10:59:08,False,IMAP,405.76,93694,2.96,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0
4,2026-08-10T10:59:00,False,ACE,417.46,61169,0.25,NaN,NaN,NaN,NaN,...,None,None,0,0,0,0,0,0,0,0


The API contains thousands of historical measurements. Since the application always predicts using the most recent conditions, only the latest observation is selected.

In [4]:
latest_wind = df_wind.iloc[0]

latest_wind

time_tag                 2026-08-10T11:01:00
active                                  True
source                                SOLAR1
proton_speed                           414.6
proton_temperature                    138877
proton_density                          3.65
proton_vx_gse                         -412.1
proton_vy_gse                           13.1
proton_vz_gse                          -43.4
proton_vx_gsm                         -412.1
proton_vy_gsm                           -8.3
proton_vz_gsm                          -44.5
proton_sample_size                         1
alpha_speed                             None
alpha_temperature                       None
alpha_density                           None
alpha_vx_gse                            None
alpha_vy_gse                            None
alpha_vz_gse                            None
alpha_vx_gsm                            None
alpha_vy_gsm                            None
alpha_vz_gsm                            None
alpha_samp

The prediction model only requires three solar wind variables. These values are extracted and stored in a dictionary that will later be combined with the remaining API data.

In [5]:
solar_wind = {
    "speed": latest_wind["proton_speed"],
    "density": latest_wind["proton_density"],
    "temperature": latest_wind["proton_temperature"]
}

solar_wind

{'speed': np.float64(414.6),
 'density': np.float64(3.65),
 'temperature': np.int64(138877)}

## Real-time Magnetic Field

In addition to solar wind measurements, the prediction model also requires information about the Interplanetary Magnetic Field (IMF). These magnetic field components play an important role in determining how the solar wind interacts with Earth's magnetosphere and, therefore, are key variables for predicting geomagnetic activity.

The following steps retrieve the latest magnetic field measurements from the NOAA SWPC API and extract only the variables required by the prediction model.

In [6]:
url = "https://services.swpc.noaa.gov/json/rtsw/rtsw_mag_1m.json"

response = requests.get(url)

data = response.json()

In [7]:
df_mag = pd.DataFrame(data)

df_mag.head()

,time_tag,active,source,range,scale,sensitivity,manual_mode,sample_size,bt,bx_gse,...,theta_gse,phi_gse,bx_gsm,by_gsm,bz_gsm,theta_gsm,phi_gsm,max_telemetry_flag,max_data_flag,overall_quality
0,2026-08-10T11:01:02,False,IMAP,None,None,None,False,60,4.83,1.09,...,-13.01,283.45,1.09,-4.54,1.13,13.59,283.46,0,0,0
1,2026-08-10T11:01:00,True,SOLAR1,None,None,None,False,60,5.12,1.62,...,0.46,288.63,1.62,-4.24,2.22,26.07,290.74,0,-9999,0
2,2026-08-10T11:00:02,False,IMAP,None,None,None,False,60,4.71,0.92,...,28.36,283.32,0.92,-2.51,3.77,54.41,292.04,0,0,0
3,2026-08-10T11:00:00,True,SOLAR1,None,None,None,False,60,5.02,1.77,...,16.59,291.75,1.77,-3.31,3.32,41.36,298.47,0,-9999,0
4,2026-08-10T10:59:02,False,IMAP,None,None,None,False,60,4.62,0.88,...,37.33,284.24,0.88,-1.84,4.05,62.81,293.16,0,0,0


In [8]:
latest_mag = df_mag.iloc[0]

latest_mag

time_tag              2026-08-10T11:01:02
active                              False
source                               IMAP
range                                None
scale                                None
sensitivity                          None
manual_mode                         False
sample_size                            60
bt                                   4.83
bx_gse                               1.09
by_gse                              -4.55
bz_gse                              -1.09
theta_gse                          -13.01
phi_gse                            283.45
bx_gsm                               1.09
by_gsm                              -4.54
bz_gsm                               1.13
theta_gsm                           13.59
phi_gsm                            283.46
max_telemetry_flag                      0
max_data_flag                           0
overall_quality                         0
Name: 0, dtype: object

In [9]:
magnetic_field = {
    "bx_gse": latest_mag["bx_gse"],
    "by_gse": latest_mag["by_gse"],
    "bz_gse": latest_mag["bz_gse"],
    "theta_gse": latest_mag["theta_gse"],
    "phi_gse": latest_mag["phi_gse"],
    "bx_gsm": latest_mag["bx_gsm"],
    "by_gsm": latest_mag["by_gsm"],
    "bz_gsm": latest_mag["bz_gsm"],
    "theta_gsm": latest_mag["theta_gsm"],
    "phi_gsm": latest_mag["phi_gsm"],
    "bt": latest_mag["bt"]
}

magnetic_field

{'bx_gse': np.float64(1.09),
 'by_gse': np.float64(-4.55),
 'bz_gse': np.float64(-1.09),
 'theta_gse': np.float64(-13.01),
 'phi_gse': np.float64(283.45),
 'bx_gsm': np.float64(1.09),
 'by_gsm': np.float64(-4.54),
 'bz_gsm': np.float64(1.13),
 'theta_gsm': np.float64(13.59),
 'phi_gsm': np.float64(283.46),
 'bt': np.float64(4.83)}

## 45-Day Space Weather Forecast

While the previous APIs provide real-time measurements, the application also incorporates forecasted space weather information. This provides the forecasted geomagnetic and solar activity used by the application for future selected dates.

The NOAA 45-Day Forecast provides long-term predictions of solar and geomagnetic conditions, which can be used to extend the application's forecasting capabilities beyond real-time observations.

In [10]:
import joblib

model = joblib.load("random_forest_model.pkl")

In [11]:
type(model)

sklearn.ensemble._forest.RandomForestRegressor

In [12]:
model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",25
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"ma

In [13]:
forecast_url = "https://services.swpc.noaa.gov/json/45-day-forecast.json"

forecast_response = requests.get(forecast_url)

forecast_data = forecast_response.json()

In [14]:
df_forecast = pd.DataFrame(forecast_data["data"])

df_forecast.head()

,time,metric,value
0,2026-08-10T00:00:00Z,ap,8
1,2026-08-10T00:00:00Z,f107,90
2,2026-08-11T00:00:00Z,ap,12
3,2026-08-11T00:00:00Z,f107,90
4,2026-08-12T00:00:00Z,ap,20


Unlike the real-time APIs, each row in this dataset represents the forecast of a specific space weather metric for a particular date.

The available forecast products are explored to identify which variables may be incorporated into the prediction engine.

In [15]:
df_forecast["metric"].unique()

<ArrowStringArray>
['ap', 'f107']
Length: 2, dtype: str

The forecast currently includes two main products:

- **Ap Index** – Forecasted geomagnetic activity.
- **F10.7 Solar Flux** – Forecasted solar radio flux, commonly used as an indicator of solar activity.

Both forecast products are separated to facilitate future integration into the application.

In [16]:
forecast_ap = df_forecast[df_forecast["metric"] == "ap"]

forecast_f107 = df_forecast[df_forecast["metric"] == "f107"]

In [17]:
forecast_ap.head()

,time,metric,value
0,2026-08-10T00:00:00Z,ap,8
2,2026-08-11T00:00:00Z,ap,12
4,2026-08-12T00:00:00Z,ap,20
6,2026-08-13T00:00:00Z,ap,5
8,2026-08-14T00:00:00Z,ap,5


In [18]:
forecast_f107.head()

,time,metric,value
1,2026-08-10T00:00:00Z,f107,90
3,2026-08-11T00:00:00Z,f107,90
5,2026-08-12T00:00:00Z,f107,92
7,2026-08-13T00:00:00Z,f107,92
9,2026-08-14T00:00:00Z,f107,95


In [19]:
forecast_data = {
    "ap_today": forecast_ap.iloc[0]["value"],
    "f107_today": forecast_f107.iloc[0]["value"]
}

forecast_data

{'ap_today': np.int64(8), 'f107_today': np.int64(90)}

These forecast variables are not direct inputs to the Random Forest model but provide additional space weather context for the final aurora visibility recommendation.

## NOAA Solar Cycle Forecast

The Sun follows an approximately 11-year activity cycle. During periods of higher solar activity, the number of sunspots increases, leading to a greater likelihood of solar eruptions and geomagnetic disturbances that can produce auroras.

The Random Forest model was trained using the smoothed sunspot number as one of its input features. Since this variable is not provided by the previous real-time NOAA endpoints, an additional NOAA Solar Cycle Forecast endpoint is used to retrieve the smoothed sunspot number required to recreate the feature set used during model training.

Endpoint used:

- solar-cycle/solar-cycle-25-predicted.json
    - Forecasted smoothed_ssn

In [20]:
import pandas as pd

url = "https://services.swpc.noaa.gov/json/solar-cycle/solar-cycle-25-predicted.json"

df_ssn = pd.read_json(url)

df_ssn.head()
df_ssn.columns

Index(['time_tag', 'smoothed_ssn', 'f10.7'], dtype='str')

In [21]:
df_ssn

,time_tag,smoothed_ssn,f10.7
0,2020-05,0.676317,67.9595
1,2020-06,1.034350,68.0823
2,2020-07,1.440060,68.2226
3,2020-08,1.934630,68.3953
4,2020-09,2.606250,68.6326
...,...,...,...
243,2040-08,0.000435,67.7301
244,2040-09,0.000379,67.7301
245,2040-10,0.000334,67.7301
246,2040-11,0.000295,67.7301


In [22]:
latest_ssn = df_ssn.iloc[0]

ssn = {
    "smoothed_ssn": latest_ssn["smoothed_ssn"]
}

ssn

{'smoothed_ssn': np.float64(0.676317)}

Only the smoothed_ssn variable is required by the prediction engine, as it matches the feature used during model training.

## Open-Meteo API

Cloud cover and visibility are key environmental factors affecting aurora observations. Even when space weather conditions are favourable, dense clouds or poor visibility can prevent the aurora from being seen.

Since weather conditions vary by location, the API requires the latitude and longitude of the selected destination to retrieve the corresponding forecast.

For demonstration purposes, the coordinates of **Tromsø (Norway)** are used throughout this notebook. In the final Streamlit application, these coordinates will be automatically obtained from the destination selected by the user.

Endpoint used:

- forecast
    - Cloud cover
    - Visibility

In [23]:
# Temporary coordinates for Tromsø (Norway)

latitude = 69.6492
longitude = 18.9553

environment_url = (
    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={latitude}"
    f"&longitude={longitude}"
    f"&current=cloud_cover,visibility"
)

environment_response = requests.get(environment_url)

environment_data = environment_response.json()

In [24]:
environment_data

{'latitude': 69.648026,
 'longitude': 18.943802,
 'generationtime_ms': 0.057578086853027344,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 16.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'cloud_cover': '%',
  'visibility': 'm'},
 'current': {'time': '2026-08-10T11:00',
  'interval': 900,
  'cloud_cover': 50,
  'visibility': 63560.0}}

The prediction engine only requires the weather variables that directly influence aurora visibility.

In [25]:
environment = {
    "cloud_cover": environment_data["current"]["cloud_cover"],
    "visibility": environment_data["current"]["visibility"]
}

Even when geomagnetic conditions are favourable, the aurora may not be visible if the sky is cloudy. Therefore, the application retrieves cloud cover and visibility from the Open-Meteo API for the selected location and date.

## Sunrise-Sunset API

Auroras can only be observed when the sky is sufficiently dark. Therefore, the application also retrieves astronomical data for the selected destination before generating a prediction.

The Sunrise-Sunset API provides:

- Sunrise
- Sunset
- Civil twilight
- Nautical twilight
- Astronomical twilight

These values are used to estimate the expected **Sky Darkness**, helping determine how naturally dark the sky will be for aurora observation.

Like the Open-Meteo API, the Sunrise-Sunset API requires the **latitude** and **longitude** of the selected destination because sunrise, sunset and twilight times depend on geographic location.

For demonstration purposes, the coordinates of Tromsø (Norway) are used throughout this notebook. In the final Streamlit application, these coordinates are obtained automatically from the destination selected by the user.

Endpoint used:

- json
    - Astronomical twilight begin
    - Astronomical twilight end

In [26]:
# Temporary coordinates for Tromsø (Norway)

latitude = 69.6492
longitude = 18.9553

sun_url = (
    f"https://api.sunrise-sunset.org/json?"
    f"lat={latitude}"
    f"&lng={longitude}"
    f"&formatted=0"
)

sun_response = requests.get(sun_url)

sun_data = sun_response.json()

In [27]:
sun_data

{'results': {'sunrise': '2026-08-10T01:15:33+00:00',
  'sunset': '2026-08-10T20:23:34+00:00',
  'solar_noon': '2026-08-10T10:49:34+00:00',
  'day_length': 68881,
  'civil_twilight_begin': '1970-01-01T00:00:01+00:00',
  'civil_twilight_end': '1970-01-01T00:00:01+00:00',
  'nautical_twilight_begin': '1970-01-01T00:00:01+00:00',
  'nautical_twilight_end': '1970-01-01T00:00:01+00:00',
  'astronomical_twilight_begin': '1970-01-01T00:00:01+00:00',
  'astronomical_twilight_end': '1970-01-01T00:00:01+00:00'},
 'status': 'OK',
 'tzid': 'UTC'}

The application uses astronomical twilight information to estimate the natural sky darkness for the selected location and date.

In [28]:
sun = {
    "sunrise": sun_data["results"]["sunrise"],
    "sunset": sun_data["results"]["sunset"]
}

sun

{'sunrise': '2026-08-10T01:15:33+00:00', 'sunset': '2026-08-10T20:23:34+00:00'}

## Geolocation API

The application requires the user's geographic coordinates to retrieve location-specific weather conditions, sunrise and sunset times, and aurora predictions.

In the final Streamlit application, coordinates are obtained automatically from the destination selected or entered by the user using the Open-Meteo Geocoding API.

Endpoint used:

Open-Meteo Geocoding API

- City search
- Latitude
- Longitude
- Country

In [29]:
geo_url = "https://ipwho.is/"

geo_response = requests.get(geo_url)

location = geo_response.json()

In [30]:
location

{'ip': '2003:106:702:d842:d032:7a82:3328:d37c',
 'success': True,
 'type': 'IPv6',
 'continent': 'Europe',
 'continent_code': 'EU',
 'country': 'Germany',
 'country_code': 'DE',
 'region': 'Land Berlin',
 'region_code': 'BE',
 'city': 'Berlin',
 'latitude': 52.5195702,
 'longitude': 13.4069037,
 'is_eu': True,
 'postal': '10178',
 'calling_code': '49',
 'capital': 'Berlin',
 'borders': 'AT,BE,CH,CZ,DK,FR,LU,NL,PL',
 'flag': {'img': 'https://cdn.ipwhois.io/flags/de.svg',
  'emoji': '🇩🇪',
  'emoji_unicode': 'U+1F1E9 U+1F1EA'},
 'connection': {'asn': 3320,
  'org': 'Deutsche Telekom AG',
  'isp': 'Deutsche Telekom AG',
  'domain': 'telekom.de'},
 'timezone': {'id': 'Europe/Berlin',
  'abbr': 'CEST',
  'is_dst': True,
  'offset': 7200,
  'utc': '+02:00'},
 'readme': 'https://ipwhois.io/docs'}

In [31]:
location = {
    "city": location["city"],
    "country": location["country"],
    "latitude": location["latitude"],
    "longitude": location["longitude"]
}

location

{'city': 'Berlin',
 'country': 'Germany',
 'latitude': 52.5195702,
 'longitude': 13.4069037}

# Prediction Engine

The previous sections demonstrated how to retrieve real-time and forecasted information from multiple external APIs.

The next step is to combine these independent data sources into a single feature set that matches the structure expected by the trained Random Forest model.

This prediction engine acts as the bridge between live environmental data and the machine learning model.

The data retrieved from the different APIs—including real-time solar wind measurements, magnetic field data, and the official NOAA Solar Cycle forecast (smoothed sunspot number)—are combined into a single dictionary. This unified structure is then transformed into the feature set expected by the trained Random Forest model.

These variables recreate the feature structure used during model training and allow the Random Forest model to estimate the current Dst index from real-time space weather conditions.

In [32]:
api_data = {}

api_data.update(solar_wind)
api_data.update(magnetic_field)
api_data.update(ssn)

api_data

{'speed': np.float64(414.6),
 'density': np.float64(3.65),
 'temperature': np.int64(138877),
 'bx_gse': np.float64(1.09),
 'by_gse': np.float64(-4.55),
 'bz_gse': np.float64(-1.09),
 'theta_gse': np.float64(-13.01),
 'phi_gse': np.float64(283.45),
 'bx_gsm': np.float64(1.09),
 'by_gsm': np.float64(-4.54),
 'bz_gsm': np.float64(1.13),
 'theta_gsm': np.float64(13.59),
 'phi_gsm': np.float64(283.46),
 'bt': np.float64(4.83),
 'smoothed_ssn': np.float64(0.676317)}

In [33]:
model_input = pd.DataFrame([api_data])

In [35]:
import joblib

model_columns = joblib.load("model_columns.pkl")

In [36]:
model_input = model_input.reindex(columns=model_columns)

## Prepare the model input

The collected API data are converted into a pandas DataFrame. This is the format required before aligning the variables with the feature structure used during model training.

In [38]:
model_input = pd.DataFrame([api_data])

model_input

,speed,density,temperature,bx_gse,by_gse,bz_gse,theta_gse,phi_gse,bx_gsm,by_gsm,bz_gsm,theta_gsm,phi_gsm,bt,smoothed_ssn
0,414.6,3.65,138877,1.09,-4.55,-1.09,-13.01,283.45,1.09,-4.54,1.13,13.59,283.46,4.83,0.676317


## Align the features with the trained model

The collected API data are reordered to match the exact feature structure used during model training. This ensures that the Random Forest model receives the variables in the expected order.

In [39]:
import joblib

model_columns = joblib.load("model_columns.pkl")

model_columns

['bx_gse',
 'by_gse',
 'bz_gse',
 'theta_gse',
 'phi_gse',
 'bx_gsm',
 'by_gsm',
 'bz_gsm',
 'theta_gsm',
 'phi_gsm',
 'bt',
 'density',
 'speed',
 'temperature',
 'smoothed_ssn']

In [40]:
model_input = model_input.reindex(columns=model_columns)

model_input

,bx_gse,by_gse,bz_gse,theta_gse,phi_gse,bx_gsm,by_gsm,bz_gsm,theta_gsm,phi_gsm,bt,density,speed,temperature,smoothed_ssn
0,1.09,-4.55,-1.09,-13.01,283.45,1.09,-4.54,1.13,13.59,283.46,4.83,3.65,414.6,138877,0.676317


## Load the trained model

The trained Random Forest model is loaded from disk. It will be used to generate a prediction based on the real-time data collected from the external APIs.

In [41]:
import joblib

model = joblib.load("random_forest_model.pkl")

## Make a prediction

The prepared input data are passed to the trained Random Forest model, which estimates the expected Dst index based on the current space weather conditions.

In [42]:
prediction = model.predict(model_input)

prediction

array([-3.44])

In [43]:
prediction = prediction[0]

print(f"Predicted Dst index: {prediction:.2f} nT")

Predicted Dst index: -3.44 nT


The Random Forest model predicts a Dst index of -3.44 nT based on the current space weather conditions.

The Dst (Disturbance Storm Time) index measures disturbances in the Earth's magnetic field:

- Values close to 0 nT indicate quiet geomagnetic conditions.
- More negative values indicate stronger geomagnetic storms, increasing the likelihood of auroral activity.

A predicted value of -3.44 nT suggests that geomagnetic activity is currently low, so favourable aurora conditions are unlikely based on the Dst prediction alone.

## Prediction Interpretation

The Random Forest model predicts the expected Dst index based on the current space weather conditions.

In the Streamlit application, the Dst prediction is presented as Today's Machine Learning Aurora Estimate, providing a real-time indication of current geomagnetic storm intensity. Separately, the application estimates the observation chance for a selected location and date using forecast Ap, latitude, sky darkness, cloud cover and visibility.

# Streamlit Integration

Let's switch to aurora_forecast_app.py

## App Logic

The application produces two complementary outputs: Today's Machine Learning Aurora Estimate and Estimated Observation Chance (weighted scoring system).

## 1. Today's Machine Learning Aurora Estimate

**Purpose:** Estimate the **current geomagnetic conditions** using Machine Learning and show the expected **short-term auroral activity**.

**How it works:**

1. Retrieves **real-time** NOAA data:

   * Solar wind
   * Magnetic field
   * Solar Cycle (SSN)
          
2. The data is fed into a **Random Forest** model.
3. The model predicts the **Dst Index**, an indicator of geomagnetic storm intensity.
4. The predicted Dst is translated into an easy-to-understand message:

* prediction > -30          → No significant activity
* -50 < prediction ≤ -30    → Faint
* -100 < prediction ≤ -50   → Moderate
* -200 < prediction ≤ -100  → Strong
* prediction ≤ -200         → Exceptional
  
More negative Dst values represent stronger geomagnetic disturbances and greater aurora potential.

The app also retrieves NOAA OVATION aurora data to display expected auroral activity for the next 30–40 minutes on an interactive map centred on the selected destination. 

## 2. Estimated Observation Chance

Purpose: Estimate how favourable the conditions are for observing the Northern Lights on the selected date.

How it works:

1. Retrieves forecast data:
- NOAA 45-Day Forecast
    - Ap Index (forecasted level of geomagnetic activity, used to calculate aurora potential)
    - F10.7 Solar Flux (contextual solar activity indicator)
- Open-Meteo
    - Cloud cover
    - Visibility
- Sunrise-Sunset
    - Sky darkness
- Selected location
    - Latitude
      
2. Calculates **aurora potential** from:
- Forecast Ap
- Latitude

3. Calculates **observation conditions** from:
- Sky darkness
- Cloud cover
- Visibility
  
4. **Combines both parts with:**

Estimated Observation Chance
=
Aurora Potential × Observation Conditions × 100

**+**

Aurora Potential
=
Ap Factor × Latitude Factor

**+**

Observation Conditions
=
Darkness Factor × Cloud Factor × Visibility Factor

**Why this approach?**

Observing the Northern Lights depends on two main components: whether auroral activity can occur at the selected location and whether atmospheric conditions allow it to be seen.

The first component is estimated from geomagnetic activity (Ap) and latitude. The second considers darkness, cloud cover and visibility. Multiplying these components produces a single Estimated Observation Chance that represents how favourable the overall conditions are for observing an aurora.